In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:

# ---------------------------------------------------------
# 1. Dataset sintetico: y = theta0 + theta1 * x + ruido
# ---------------------------------------------------------
rng = np.random.default_rng(42)
n = 30
true_theta0, true_theta1 = 3.0, 1.2
x = rng.uniform(0, 10, n)
y = true_theta0 + true_theta1 * x + rng.normal(0, 1.4, n)
 
# Matriz de diseno X con columna de 1s (bias/intercepto)
X = np.column_stack([np.ones(n), x])   # shape (n, 2)
 
 
# ---------------------------------------------------------
# 2. Funcion de costo J(theta) y su gradiente (forma vectorizada)
# ---------------------------------------------------------
def cost(theta, X, y):
    m = len(y)
    errors = X @ theta - y
    return (errors @ errors) / (2 * m)
 
 
def gradient(theta, X, y):
    m = len(y)
    return (X.T @ (X @ theta - y)) / m
 
 
# ---------------------------------------------------------
# 3. Batch gradient descent, guardando el camino completo
# ---------------------------------------------------------
alpha = 0.03
n_iters = 400
theta = np.array([-1.5, -0.5])  # punto de partida deliberadamente lejos del optimo
 
path = [theta.copy()]
for _ in range(n_iters):
    theta = theta - alpha * gradient(theta, X, y)
    path.append(theta.copy())
path = np.array(path)          # shape (n_iters+1, 2)
costs = np.array([cost(t, X, y) for t in path])
 
print(f"theta inicial: {path[0]}")
print(f"theta final:   {path[-1]}  (verdadero: [{true_theta0}, {true_theta1}])")
print(f"J inicial: {costs[0]:.4f}  ->  J final: {costs[-1]:.4f}")
 
# ---------------------------------------------------------
# 4. Grilla para la superficie J(theta0, theta1)
# ---------------------------------------------------------
t0_range = np.linspace(-2.5, 8, 60)
t1_range = np.linspace(-1.5, 3, 60)
T0, T1 = np.meshgrid(t0_range, t1_range)
Z = np.zeros_like(T0)
for i in range(T0.shape[0]):
    for j in range(T0.shape[1]):
        Z[i, j] = cost(np.array([T0[i, j], T1[i, j]]), X, y)
 
# ---------------------------------------------------------
# 5. Figura con 3 paneles
# ---------------------------------------------------------
fig = plt.figure(figsize=(15, 5))
ax_fit = fig.add_subplot(1, 3, 1)
ax_surf = fig.add_subplot(1, 3, 2, projection="3d")
ax_cont = fig.add_subplot(1, 3, 3)
 
# --- Panel 1: scatter + recta ---
ax_fit.scatter(x, y, color="gray", s=25, zorder=3, label="datos")
x_line = np.array([x.min(), x.max()])
line_fit, = ax_fit.plot([], [], color="tab:blue", lw=2.5, label="h(x) = theta0 + theta1 x")
ax_fit.set_xlabel("x")
ax_fit.set_ylabel("y")
ax_fit.set_title("Ajuste del modelo lineal")
ax_fit.legend(loc="upper left", fontsize=8)
 
# --- Panel 2: superficie 3D ---
ax_surf.plot_surface(T0, T1, Z, cmap="Blues", alpha=0.7, linewidth=0, antialiased=True)
point_surf, = ax_surf.plot([], [], [], color="tab:red", marker="o", markersize=5)
path_surf, = ax_surf.plot([], [], [], color="tab:red", lw=1.5)
ax_surf.set_xlabel("theta0")
ax_surf.set_ylabel("theta1")
ax_surf.set_zlabel("J(theta)")
ax_surf.set_title("Superficie convexa de J")
ax_surf.view_init(elev=30, azim=-60)
 
# --- Panel 3: curvas de nivel (contour) ---
ax_cont.contour(T0, T1, Z, levels=25, cmap="Blues")
point_cont, = ax_cont.plot([], [], color="tab:red", marker="o", markersize=5)
path_cont, = ax_cont.plot([], [], color="tab:red", lw=1.5)
ax_cont.set_xlabel("theta0")
ax_cont.set_ylabel("theta1")
ax_cont.set_title("Curvas de nivel y camino del descenso")
 
fig.suptitle("Batch gradient descent: iteracion 0", fontsize=13)
fig.tight_layout()
 
 
